# Scientific Article Recommendation System
**02807 Computational Tools for Data Science**

**Authots** - Group 69
- Yann BECKER - s253048
- Pierre-Eduard KOLAR - s254145
- Pierre HOLLEBEQUE - s254136
- Thibaut HEIM - s252933

This notebook contains the code associated with the project report. It consists of a first part on preprocessing and a second part where the methods are implemented and evaluated.

---

# I - Preprocessing
This part implements a preprocessing pipeline for the arXiv dataset. Its primary goal is to standardize article identifiers, clean textual data (titles, abstracts), and integrate citation graph information into a unified JSON structure. It will be used to compare three recommendation methods.

**Key Objectives:**
1.  **ID Normalization:** Convert all arXiv IDs (including legacy formats like `alg-geom/9202013`) into a consistent, digits-only format to ensure reliable matching across datasets.
2.  **Metadata Parsing:** Load and clean article metadata (titles, authors, categories, abstracts).
3.  **Graph Integration:** Merge the metadata with an external citation graph, resolving references to the standardized IDs.
4.  **Gold Dataset Creation:** Generate specialized evaluation datasets ("Gold Datasets") by downloading source LaTeX files and extracting ground-truth citation contexts.


## Part 1 : Data Unification

### 1.1 Environment Setup & Configuration

We begin by importing necessary libraries and defining file paths. The configuration block allows for easy adjustment of input/output directories.

To run the code, you must place the JSON files mentioned in the report in a `data` folder.

In [ ]:
import json
import gzip
import sys
import os
from typing import Optional

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

METADATA_FILE = "data/arxiv-metadata-oai-snapshot.json"
GRAPH_FILE = "data/internal-references-pdftotext.json"

OUTPUT_FILE = "data/processed/unified_articles.json"

METADATA_ID_KEY = "id"
METADATA_TITLE_KEY = "title"
METADATA_ABSTRACT_KEY = "abstract"
METADATA_AUTHORS_KEY = "authors"
METADATA_CATEGORIES_KEY = "categories"

### 1.2 Data Normalization Helpers

These utility functions are the core of our data cleaning strategy.

*   `normalize_to_simple_id`: Handles the complexity of arXiv's changing ID formats over the last 30+ years, stripping everything down to a purely numeric string.
*   `clean_text`: Prepares raw text (like abstracts) for NLP tasks by removing HTML tags, special characters, and normalizing whitespace.

In [ ]:
# ============================================================================
# NORMALIZATION FUNCTION 
# ============================================================================

def normalize_to_simple_id(id_str: str) -> str:
    """
    Normalize any ID format to a simple digits-only string.

    Examples:
    '9202.013' -> '9202013'
    'alg-geom/9202013' -> '9202013'
    'cs.LG/210100001' -> '210100001'
    '0001.234' -> '0001234'

    Works for both Kaggle and Graph formats.

    Args:
    id_str: ID string in Kaggle or Graph format

    Returns:
    Normalized ID containing digits only.
    """
    # Keep only digit characters to create the canonical ID used across datasets
    return ''.join(c for c in id_str if c.isdigit())

In [ ]:
# ============================================================================
# PREPROCESSING
# ============================================================================

import re

# Regex precompiled for performance
HTML_TAG_REGEX = re.compile(r"<.*?>")
SPECIAL_CHAR_REGEX = re.compile(r"[^a-zA-Z0-9\s]")

def clean_text(text: str) -> str:
    """Clean abstract text for NLP tasks"""
    if not text:
        return ""
    
    text = HTML_TAG_REGEX.sub("", text)
    text = text.replace("\\n", " ").replace("\\", "")
    text = SPECIAL_CHAR_REGEX.sub("", text)
    text = text.lower().strip()
    
    return text

### 1.3 Main Unified Dataset Builder

This function orchestrates the entire unification process in four steps:
1.  **Load Metadata:** Reads the massive JSONL metadata file, normalizing every ID and filtering out articles with empty abstracts.
2.  **Load Graph:** Reads the citation graph, ensuring all source and target IDs are normalized to match the metadata.
3.  **Attach Citations:** Links the two datasets. For every article in the metadata, we look up its citations in the graph, validate that the cited papers actually exist in our dataset, and attach them.
4.  **Save:** Writes the final, enriched dataset to disk.

In [ ]:
# ============================================================================
# MAIN PROCESSING
# ============================================================================

def build_unified_dataset(metadata_path: str, 
                          graph_path: str, 
                          output_path: str):
    """
    Build a unified dataset using simplified numeric-only IDs for internal consistency.
    The function reads the metadata stream, normalizes IDs, builds lookup sets,
    normalizes the citation graph IDs, attaches valid citations, and saves the final JSON.
    """
    
    print("="*80)
    print("UNIFIED PREPROCESSING - SIMPLE ID FORMAT")
    print("="*80)

    # Step 1: Load metadata and normalize IDs
    print("\n[Step 1] Loading metadata and normalizing IDs...")
    
    all_ids = set() # normalized id set
    all_articles = {} # dict keyed by normalized id
    original_id_map = {} # mapping normalized_id -> original id (for output)
    
    papers_processed = 0
    papers_kept = 0
    
    try:
        f_open = gzip.open if metadata_path.endswith('.gz') else open
        
        with f_open(metadata_path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    article = json.loads(line)
                    
                    # original id
                    original_id = article.get(METADATA_ID_KEY)
                    if not original_id:
                        continue
                    
                    # Normalize id
                    normalized_id = normalize_to_simple_id(original_id)
                    
                    # Add to id set
                    all_ids.add(normalized_id)
                    
                    papers_processed += 1
                    
                    if papers_processed % 100000 == 0:
                        print(f" ... {papers_processed:,} processed, {papers_kept:,} kept")
                    
                    # Extract fields
                    title = article.get(METADATA_TITLE_KEY, "")
                    abstract = article.get(METADATA_ABSTRACT_KEY, "")
                    authors = article.get(METADATA_AUTHORS_KEY, [])
                    categories = article.get(METADATA_CATEGORIES_KEY, "")
                    
                    # Filters: keep only meaningful abstracts
                    if not abstract:
                        continue
                    
                    clean_abstract = clean_text(abstract)
                    
                    if len(clean_abstract) < 50:
                        continue
                    
                    # Store using normalized ID as the key
                    all_articles[normalized_id] = {
                        "id": original_id, # keep original Kaggle ID for the output
                        "normalized_id": normalized_id, # for debugging
                        "title": title,
                        "authors": authors if isinstance(authors, list) else [],
                        "abstract": abstract,
                        "clean_text": clean_abstract,
                        "categories": categories.strip() if categories else "",
                        "refs": [] # to be filled later
                    }
                    
                    original_id_map[normalized_id] = original_id
                    papers_kept += 1
                    
                except json.JSONDecodeError:
                    # skip invalid JSON lines
                    continue
                    
        print(f"\nStep 1 finished:")
        print(f" - Papers processed: {papers_processed:,}")
        print(f" - Papers kept: {papers_kept:,}")
        print(f" - Unique normalized IDs: {len(all_ids):,}")
        
    except FileNotFoundError:
        print(f"ERROR: '{metadata_path}' not found.")
        sys.exit(1)
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)
        
    # Step 2: Load citation graph and normalize IDs
    print("\n[Step 2] Loading citation graph and normalizing IDs...")
    
    try:
        f_open = gzip.open if graph_path.endswith('.gz') else open
        
        print(" Loading graph file...")
        with f_open(graph_path, 'rt', encoding='utf-8') as f:
            citation_graph_raw = json.load(f)
            
        print(f" Graph loaded: {len(citation_graph_raw):,} papers")
        
        # Normalize all ids in the graph (source and targets)
        print(" Normalizing graph IDs...")
        citation_graph_normalized = {}
        
        for source_id_raw, refs_raw in citation_graph_raw.items():
            source_id_norm = normalize_to_simple_id(source_id_raw)
            
            refs_normalized = []
            if isinstance(refs_raw, list):
                refs_normalized = [normalize_to_simple_id(ref) for ref in refs_raw]
            
            citation_graph_normalized[source_id_norm] = refs_normalized
            
        print(f" {len(citation_graph_normalized):,} normalized IDs in graph")
        
    except FileNotFoundError:
        print(f"ERROR: '{graph_path}' not found.")
        sys.exit(1)
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)
        
    # Step 3: Attach citations (simple match using normalized ids)
    print("\n[Step 3] Adding citations...")
    
    refs_added = 0
    refs_total = 0
    refs_valid = 0
    
    papers_checked = 0
    
    for normalized_id, article_data in all_articles.items():
        papers_checked += 1
        
        if papers_checked % 50000 == 0:
            print(f" ... {papers_checked:,}/{len(all_articles):,} papers")
        
        # direct lookup in the normalized graph
        if normalized_id in citation_graph_normalized:
            refs_normalized = citation_graph_normalized[normalized_id]
            
            refs_total += len(refs_normalized)
            
            # keep only refs that exist in metadata (avoid dangling refs)
            valid_refs = [ref for ref in refs_normalized if ref in all_ids]
            
            refs_valid += len(valid_refs)
            
            # deduplicate
            unique_refs = list(set(valid_refs))
            
            # map back to original Kaggle IDs for output
            original_refs = [original_id_map.get(ref, ref) for ref in unique_refs]
            
            article_data["refs"] = original_refs
            
            if original_refs:
                refs_added += 1
                
    print(f"\nStep 3 finished:")
    print(f" - Papers with citations: {refs_added:,}")
    print(f" - Total references (raw): {refs_total:,}")
    print(f" - Valid references: {refs_valid:,}")
    
    if len(all_articles) > 0:
        coverage = (refs_added / len(all_articles)) * 100
        print(f" - Coverage: {coverage:.1f}%")
        
    # Step 4: Save output JSON
    print("\n[Step 4] Saving output...")
    
    try:
        output_dir = os.path.dirname(output_path)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
            
        # Remove debug field 'normalized_id' before saving
        for article in all_articles.values():
            article.pop('normalized_id', None)
            
        final_articles = list(all_articles.values())
        
        stats = {
            'papers_with_refs': sum(1 for a in final_articles if a["refs"]),
            'papers_without_refs': sum(1 for a in final_articles if not a["refs"]),
            'total_refs': sum(len(a["refs"]) for a in final_articles)
        }
        
        final_data = {
            "articles": final_articles,
            "metadata": {
                "total_papers": len(final_articles),
                "papers_with_refs": stats['papers_with_refs'],
                "papers_without_refs": stats['papers_without_refs'],
                "total_references": stats['total_refs'],
                "has_clean_text": True,
                "has_citations": True,
                "has_categories": True,
                "id_format": "Kaggle original format",
                "normalization": "digits only internally"
            }
        }
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(final_data, f, indent=2)
            
        print(f"\nDONE. Dataset saved to: {output_path}")
        print(f"\nFinal statistics:")
        print(f" - Total papers: {len(final_articles):,}")
        print(f" - Papers with references: {stats['papers_with_refs']:,}")
        print(f" - Papers without references: {stats['papers_without_refs']:,}")
        print(f" - Total references: {stats['total_refs']:,}")
        
        if stats['papers_with_refs'] > 0:
            avg_refs = stats['total_refs'] / stats['papers_with_refs']
            print(f" - Avg refs per paper: {avg_refs:.2f}")
            
    except Exception as e:
        print(f"ERROR while saving: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

**Execution**
Run the unified dataset builder with the paths configured above.

In [ ]:
build_unified_dataset(
    METADATA_FILE,
    GRAPH_FILE,
    OUTPUT_FILE
)

## Part2 : Gold Dataset Creation & Subdataset Cleaning Pipeline
This section implements the end-to-end pipeline for generating "Gold Datasets"—subsets of articles where citations are explicitly linked to their textual context using full LaTeX source files.

**Pipeline Steps:**
1.  **Load** the unified arXiv dataset created above.
2.  **Generate Candidates** using three distinct strategies: Water-filling (Stratified), Most Cited (Top-N), and Quartile-based.
3.  **Create Gold Datasets**: For each strategy, download LaTeX source files, resolve citation keys (e.g., `\cite{alexnet}`) to arXiv IDs, and extract the surrounding text.
4.  **Clean Subdatasets**: Produce final training datasets by removing the Gold articles to prevent data leakage.
5.  **Save** all artifacts in structured JSON.

In [ ]:
import json
import os
import re
import tarfile
import gzip
import shutil
import requests
from typing import List, Dict, Set, Any, Tuple, Optional
from collections import defaultdict
from difflib import SequenceMatcher

### 2.1 Gold Dataset Configuration

Setup of input/output paths and regex patterns used to parse LaTeX files. `TARGET_GOLD_SIZE` determines how many articles with resolved citations we aim to collect per strategy.

In [ ]:
DATA_PATH = 'data/processed/unified_articles.json'
OUTPUT_DIR = 'data/gold_datasets_linked/'
TEMP_DIR = 'temp_latex_downloads_linked/'
TARGET_GOLD_SIZE = 20
CONTEXT_WINDOW = 400

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
if not os.path.exists(TEMP_DIR):
    os.makedirs(TEMP_DIR)

# --- Pre-compiled Regex ---
# Matches LaTeX citations like \cite{foo}, \citep{foo, bar}
CITE_PATTERN = re.compile(r'(\\(cite|citep|citet|bibcite)\{([^}]+)\})')

# Matches bibliography items: \bibitem{key} The Title ...
BIBITEM_PATTERN = re.compile(r'\\bibitem\{([^}]+)\}([\s\S]*?)(?=\\bibitem|\\end\{thebibliography\})')

def clean_latex_text(text: str) -> str:
    """Cleans LaTeX formatting for readability."""
    if not text: return ""
    # Remove comments
    text = re.sub(r'%.*', '', text)
    # Simplify common commands
    text = re.sub(r'\\(textbf|textit|emph|section|subsection)\{([^}]+)\}', r'\2', text)
    text = re.sub(r'\$([^$]+)\$', r'\1', text) # Inline math
    # Normalize whitespace
    return re.sub(r'\s+', ' ', text).strip()

### 2.2 Data Indexing

We build a global index of `ID -> Normalized Title`. This is crucial for the fuzzy matching step, where we check if a title found in a paper's LaTeX bibliography corresponds to a known arXiv ID in our dataset.

In [ ]:
def load_and_index_data(filepath: str) -> Tuple[List[Dict], Dict[str, str]]:
    """
    Loads data and creates a global mapping of ID -> Title.
    This allows us to look up the titles of referenced IDs.
    """
    print(f"Loading dataset from {filepath}...")
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    articles = data['articles'] if isinstance(data, dict) and 'articles' in data else data
    
    print("Building ID -> Title index...")
    id_to_title = {}
    for art in articles:
        if 'id' in art and 'title' in art:
            # Normalize title for better matching (lowercase, no punctuation)
            clean_t = re.sub(r'[^a-z0-9\s]', '', art['title'].lower())
            id_to_title[art['id']] = clean_t
            
    print(f"Indexed {len(id_to_title)} titles.")
    return articles, id_to_title

articles_data, global_id_to_title = load_and_index_data(DATA_PATH)

### 2.3 Source File Extraction

These functions handle the retrieval and extraction of raw source files from arXiv.
*   `download_source`: Downloads the `.tar.gz` package.
*   `extract_source_content`: Unpacks the archive and concatenates all `.tex` files into a single body string. It also separates the bibliography (either from `.bbl` files or embedded `\thebibliography`).

In [ ]:
def download_source(arxiv_id: str, save_dir: str) -> Optional[str]:
    """Downloads .tar.gz source from arXiv."""
    url = f"https://arxiv.org/e-print/{arxiv_id}"
    save_path = os.path.join(save_dir, f"{arxiv_id}.tar.gz")
    try:
        response = requests.get(url, stream=True, timeout=20)
        if response.status_code == 200 and 'pdf' not in response.headers.get('content-type', ''):
            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            return save_path
    except Exception as e:
        print(f"Error downloading {arxiv_id}: {e}")
        return None

def extract_source_content(file_path: str) -> Tuple[str, str]:
    """
    Extracts:
    1. Full combined LaTeX body (for text extraction)
    2. Combined Bibliography content (from .bbl files or embedded \thebibliography)
    """
    full_body = ""
    full_bib = ""
    
    try:
        if tarfile.is_tarfile(file_path):
            with tarfile.open(file_path, 'r:*') as tar:
                for member in tar.getmembers():
                    # Extract Text
                    if member.name.endswith('.tex'):
                        try:
                            f = tar.extractfile(member)
                            content = f.read().decode('utf-8', errors='ignore')
                            full_body += f"\n% --- FILE: {member.name} ---\n{content}"
                            # Check for embedded bibliography
                            if '\\begin{thebibliography}' in content:
                                full_bib += content
                        except: pass
                    
                    # Extract Bibliography file
                    elif member.name.endswith('.bbl'):
                        try:
                            f = tar.extractfile(member)
                            full_bib += f.read().decode('utf-8', errors='ignore') + "\n"
                        except: pass
        else:
            # Single file case
            with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                full_body = content
                full_bib = content
                
    except Exception as e:
        print(f"Error extracting {file_path}: {e}")
        
    return full_body, full_bib

### 2.4 Citation Resolution Logic

This component performs the "Linkage" in our pipeline.
1.  **Parse:** We read the extracted bibliography to find pairs of `(Citation Key, Paper Title)`. E.g., `('alexnet', 'ImageNet Classification with Deep CNNs...')`.
2.  **Resolve:** We take the list of actual arXiv IDs that the metadata says this paper cites. We fuzzy-match the parsed titles against the titles of these known IDs. If a match is strong (>0.65), we map the local key `alexnet` to the global ID `1207.0560`.

In [ ]:
def parse_bibliography_keys(bib_content: str) -> Dict[str, str]:
    """
    Parses \bibitem{key} ... entries.
    Returns map: { cite_key: raw_bib_text }
    """
    bib_map = {}
    for match in BIBITEM_PATTERN.finditer(bib_content):
        key = match.group(1).strip()
        text = match.group(2)
        # Clean up the text to find a title
        # We assume the title is the longest segment or use a simple heuristic
        clean_text = re.sub(r'\\(newblock|emph|textit|textbf)', ' ', text)
        clean_text = re.sub(r'[^a-zA-Z0-9\s]', '', clean_text.lower())
        bib_map[key] = clean_text
    return bib_map

def resolve_citations(bib_map: Dict[str, str], known_ref_ids: List[str]) -> Dict[str, str]:
    """
    Links citation keys to real ArXiv IDs using fuzzy title matching.
    
    Args:
    bib_map: {cite_key: bib_text_from_latex}
    known_ref_ids: List of ArXiv IDs that this paper actually cites (from dataset)
    
    Returns:
    key_to_id: {cite_key: arxiv_id}
    """
    key_to_id = {}
    
    # 1. Retrieve titles for known reference IDs
    candidate_titles = {}
    for rid in known_ref_ids:
        if rid in global_id_to_title:
            candidate_titles[rid] = global_id_to_title[rid]

    if not candidate_titles:
        return {}

    # 2. Match keys to IDs
    for key, bib_text in bib_map.items():
        best_score = 0.0
        best_id = None
        
        # Optimization: Bib text is usually long. Title is a substring.
        # We check if the candidate title exists roughly inside the bib text.
        for rid, title in candidate_titles.items():
            # Fast check: is title subset of bib text?
            if title in bib_text:
                score = 1.0
            else:
                # Slow check: SequenceMatcher
                # We only match first 200 chars of bib entry to save time
                score = SequenceMatcher(None, title, bib_text[:300]).ratio()
            
            if score > best_score:
                best_score = score
                best_id = rid
        
        # Threshold for a valid match
        if best_score > 0.65: 
            key_to_id[key] = best_id
            
    return key_to_id

### 2.5 Subdataset Generation Strategies

We define three methods to select candidate articles for our subdatasets. All methods return a dictionary wrapper `{"articles": [...]}`.

1.  **Most Cited:** Simple Top-N based on reference count.
2.  **Stratified (Simple):** Uniform sampling across categories. *This method is naive and does not allow us to accurately predict the size of the desired subdataset. It serves only as a comparison with the improved version (Water Filling)*.
3.  **Balanced Quartiles:** Ensures representation from all tiers of citation impact (Top 25% to Bottom 25%).
4.  **Water Filling:** A robust stratified approach that handles uneven category sizes by filling quotas from larger categories when smaller ones run out.

In [ ]:
def get_candidates_most_cited(articles: List[Dict], top_n: int = 50000) -> List[Dict]:
    """Returns top N articles sorted by number of references (or citations if available)."""
    # Sorting by number of OUTGOING references as a proxy if citation count isn't in metadata.
    # If you have an 'citations_count' field, use that instead.
    # Assuming 'refs' field exists from preprocessing.
    sorted_articles = sorted(articles, key=lambda x: len(x.get('refs', [])), reverse=True)
    return sorted_articles[:top_n]

In [ ]:
def get_candidates_stratified(articles: List[Dict], total_k: int = 50000) -> List[Dict]:
    """Returns K articles distributed across categories."""
    # 1. Group by category
    cat_map = defaultdict(list)
    for art in articles:
        cats = art.get('categories', '').split()
        if cats:
            primary_cat = cats[0]
            cat_map[primary_cat].append(art)
            
    # 2. Calculate target per category
    n_cats = len(cat_map)
    if n_cats == 0: return []
    target_per_cat = max(1, total_k // n_cats)
    
    selected = []
    # 3. Select top cited from each category
    for cat, arts in cat_map.items():
        # Sort by refs count
        sorted_arts = sorted(arts, key=lambda x: len(x.get('refs', [])), reverse=True)
        selected.extend(sorted_arts[:target_per_cat])
        
    return selected[:total_k]

In [ ]:
def get_candidates_balanced_quartiles(articles: List[Dict], total_n: int = 50000) -> List[Dict]:
    """
    Creates a subdataset of size `total_n` composed of equal shares from each 
    citation quartile of the original dataset.
    
    - 25% of `total_n` from Q1 (Top 25% most cited)
    - 25% of `total_n` from Q2
    - 25% of `total_n` from Q3
    - 25% of `total_n` from Q4 (Bottom 25% least cited)
    """
    # 1. Sort all articles by citation count (Descending: Most cited -> Least cited)
    # Using 'refs' length as proxy for citations if 'citations_count' is unavailable
    sorted_articles = sorted(articles, key=lambda x: len(x.get('refs', [])), reverse=True)
    
    total_source = len(sorted_articles)
    if total_source == 0:
        return []

    # 2. Define the size of one quartile in the SOURCE dataset
    source_q_size = total_source // 4
    
    # 3. Define how many items we want from each quartile in the OUTPUT
    target_per_quartile = total_n // 4
    
    balanced_selection = []
    
    # 4. Iterate through the 4 quartiles
    for q in range(4):
        # Define start/end indices for this quartile in the sorted source list
        start_idx = q * source_q_size
        
        # For the last quartile, ensure we go to the very end (handle remainder)
        if q == 3:
            end_idx = total_source
        else:
            end_idx = start_idx + source_q_size
        
        # Extract the full pool for this quartile
        quartile_pool = sorted_articles[start_idx:end_idx]
        
        # Select the top portion of this specific quartile to meet our target
        # (Since pool is sorted, this picks the 'best' of the quartile. 
        # Use random.sample(quartile_pool, target_per_quartile) if you want random sampling within the quartile)
        selection = quartile_pool[:target_per_quartile]
        
        balanced_selection.extend(selection)
        
    # 5. Handle Rounding Errors (if total_n isn't divisible by 4)
    # Fill any remaining slots with the highest cited papers not yet selected (from Q1)
    missing = total_n - len(balanced_selection)
    if missing > 0:
        # We took the first 'target_per_quartile' from Q1. 
        # The next best candidates start immediately after that index.
        remainder_start = target_per_quartile
        remainder_end = remainder_start + missing
        
        # Ensure we don't go out of bounds of Q1
        if remainder_end < source_q_size:
            balanced_selection.extend(sorted_articles[remainder_start:remainder_end])
    
    return balanced_selection

In [ ]:
from typing import List, Dict, Any, Set
from collections import defaultdict
import networkx as nx
import math

def get_candidates_stratified_waterfilling(articles: List[Dict], top_n: int = 50000) -> Dict[str, List[Dict]]:
    """
    Selects K articles using a Water Filling (Cascade) approach to balance categories.
    Prioritizes highly cited articles within each category (using in-degree).
    
    Returns:
    Dict with structure {"articles": [selected_article_dicts]}
    """
    print(f"Starting Cascade Sampling (Target: {top_n} articles)...")
    
    # 1. Create the graph and the Primary Category dictionary
    print(f"Building graph and mapping primary categories for {len(articles)} articles...")
    
    primary_category_dict: Dict[str, List[str]] = {}
    G_full = nx.DiGraph()
    article_id_to_data = {} 
    
    for article in articles:
        article_id = article.get('id')
        
        if article_id is not None:
            # Extract primary category
            cat_string = article.get('categories', '')
            article_cats = cat_string.split(" ") if cat_string else []
            primary_cat = article_cats[0] if article_cats else "unknown"
            
            # Build Graph for scoring (in-degree calculation)
            G_full.add_node(article_id) 
            article_id_to_data[article_id] = article
            for link in article.get('refs', []):
                G_full.add_edge(article_id, link)
            
            # Map article to its PRIMARY category
            if primary_cat not in primary_category_dict:
                primary_category_dict[primary_cat] = []
            primary_category_dict[primary_cat].append(article_id)

    # 2. Prepare for Selection
    final_node_ids = set()
    
    # Sort categories by size (largest first)
    sorted_categories = sorted(
        primary_category_dict.keys(), 
        key=lambda k: len(primary_category_dict[k]), 
        reverse=True
    )
    
    nb_categories = len(sorted_categories)
    if nb_categories == 0:
        return {"articles": []}

    # Calculate initial target per category
    target_per_cat = int(math.ceil(top_n / nb_categories))
    print(f"Targeting approx. {target_per_cat} articles per category across {nb_categories} categories...")

    # Pre-calculate degrees for O(1) access
    in_degrees = dict(G_full.in_degree())
    
    # Store sorted articles per category to avoid re-sorting in Pass 2
    sorted_articles_map = {} 

    # --- PASS 1: The "Fair Share" Allocation ---
    # Take up to 'target_per_cat' from each category, sorted by quality (in-degree)
    for cat in sorted_categories:
        article_ids = primary_category_dict[cat]
        
        # Sort by in-degree (highest first)
        sorted_articles = sorted(
            article_ids, 
            key=lambda x: in_degrees.get(x, 0), 
            reverse=True
        )
        sorted_articles_map[cat] = sorted_articles # Save for Pass 2
        
        # Take the quota
        take_count = min(target_per_cat, len(sorted_articles))
        selected_subset = sorted_articles[:take_count]
        
        final_node_ids.update(selected_subset)
        
    print(f"After Pass 1: Selected {len(final_node_ids)} articles.")

    # --- PASS 2: The "Fill Remaining" (Water Filling) ---
    # If we haven't reached top_n (because some categories were small), fill from the big ones
    if len(final_node_ids) < top_n:
        remaining_needed = top_n - len(final_node_ids)
        print(f"Pass 2: Filling gap of {remaining_needed} articles from remaining pools...")
        
        for cat in sorted_categories:
            if remaining_needed <= 0:
                break
            
            all_cat_articles = sorted_articles_map[cat]
            
            # Identify articles NOT yet selected
            # We already took the first 'take_count' (calculated above)
            already_taken_count = min(target_per_cat, len(all_cat_articles))
            
            candidates = all_cat_articles[already_taken_count:]
            
            if not candidates:
                continue
            
            # Take as many as needed or available
            take_extra = min(remaining_needed, len(candidates))
            
            final_node_ids.update(candidates[:take_extra])
            remaining_needed -= take_extra

    print(f"Total unique articles selected: {len(final_node_ids)}")

    # 3. Return Articles
    print("Finalizing candidate list...")
    
    filtered_articles = []
    for node_id in final_node_ids:
        if node_id in article_id_to_data:
            filtered_articles.append(article_id_to_data[node_id])

    return {"articles": filtered_articles}

### 2.6 Processing Pipeline Execution

This block ties everything together. It defines `process_article_for_gold` (the per-article worker) and `run_gold_creation_pipeline` (the manager that loops through strategies).

1.  **Candidates:** Wrapper dictionary is generated.
2.  **Save Subdataset:** The full set of candidates is saved immediately.
3.  **Gold Extraction:** We iterate through the candidates to find valid Gold articles.
4.  **Save Gold:** The results are saved separately.

In [ ]:
def process_article_for_gold(article: Dict) -> Optional[Dict]:
    """
    Full pipeline for a single article.
    Returns dict {id: ..., chunks: {ref_id: [text, text]}} if successful.
    """
    aid = article['id']
    known_refs = article.get('refs', [])
    if not known_refs:
        return None

    # 1. Download
    source_path = download_source(aid, TEMP_DIR)
    if not source_path: return None
    
    # 2. Extract
    tex_body, bib_content = extract_source_content(source_path)
    
    # 3. Parse & Resolve Keys
    bib_keys_map = parse_bibliography_keys(bib_content)
    key_to_arxiv_id = resolve_citations(bib_keys_map, known_refs)
    
    if not key_to_arxiv_id:
        # Cleanup
        if os.path.exists(source_path): os.remove(source_path)
        return None
    
    # 4. Extract Chunks linked to IDs
    # Structure: { ref_id: [chunk1, chunk2] }
    chunks_by_id = defaultdict(list)
    
    for match in CITE_PATTERN.finditer(tex_body):
        cite_keys_str = match.group(3)
        # Handle multiple keys: \cite{key1, key2}
        keys = [k.strip() for k in cite_keys_str.split(',')]
        
        # Get context
        start, end = match.span()
        context_start = max(0, start - CONTEXT_WINDOW)
        context_end = min(len(tex_body), end + CONTEXT_WINDOW)
        raw_text = tex_body[context_start:context_end]
        clean_text = clean_latex_text(raw_text)
        
        # Assign this text to every valid ID found in the keys
        for k in keys:
            if k in key_to_arxiv_id:
                target_id = key_to_arxiv_id[k]
                chunks_by_id[target_id].append(clean_text)
                
    # Cleanup
    if os.path.exists(source_path): os.remove(source_path)
    
    if chunks_by_id:
        return {
            "id": aid,
            "citations": dict(chunks_by_id) # Convert defaultdict to dict for JSON
        }
    return None

def run_gold_creation_pipeline(strategies: Dict, articles_data: List[Dict]):
    """
    Executes the pipeline for multiple strategies.
    Expects 'strat_func' to return a dict: {"articles": [list_of_articles]}
    """
    for strat_name, strat_func in strategies.items():
        print(f"\n=== Running strategy: {strat_name} ===")
        
        # 1. Generate Candidates (Returns Dict {'articles': [...]})
        candidates_wrapper = strat_func(articles_data)
        
        # 2. Save Full Subdataset (Preserves structure {"articles": ...})
        subdataset_path = os.path.join(OUTPUT_DIR, f'subdataset_{strat_name}.json')
        with open(subdataset_path, 'w', encoding='utf-8') as f:
            json.dump(candidates_wrapper, f, indent=2)
            
        # Extract the list for processing
        candidate_list = candidates_wrapper.get("articles", [])
        print(f"Saving {len(candidate_list)} candidate articles to {subdataset_path}")
        
        # 3. Create Gold Dataset
        gold_dataset = []
        gold_ids_to_remove = []
        
        print(f"Processing candidates to find {TARGET_GOLD_SIZE} valid Gold articles...")
        
        for art in candidate_list:
            if len(gold_dataset) >= TARGET_GOLD_SIZE:
                break
                
            result = process_article_for_gold(art)
            if result:
                gold_dataset.append(result)
                gold_ids_to_remove.append(art['id'])
                print(f"✅ Added {art['id']} ({len(result['citations'])} resolved refs)")
            else:
                # Optional: Print less frequency to reduce clutter
                print(f"❌ Skipped {art['id']} (No parsed refs)")
                
        # 4. Save Gold Dataset
        gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_{strat_name}.json')
        with open(gold_path, 'w', encoding='utf-8') as f:
            json.dump(gold_dataset, f, indent=2)
            
        print(f"Done with {strat_name}. Gold dataset saved to {gold_path}")

**Select Strategies**
Uncomment the strategies you wish to run.

In [ ]:
strategies = {
    # "quartiles": lambda data: get_candidates_balanced_quartiles(data, total_n=50000),
    # "most_cited": lambda data: get_candidates_most_cited(data, top_n=50000),
    # "stratified": lambda data: get_candidates_stratified(data, total_k=50000),
    "waterfilling": lambda data: get_candidates_stratified_waterfilling(data, top_n=50000),
}

In [ ]:
# ==========================================
# EXECUTION PHASE
# ==========================================
run_gold_creation_pipeline(strategies, articles_data)

### 2.7 Subdataset Cleaning & Finalization

Once the Gold Datasets are created, we must remove those articles from the main subdatasets to ensure our training/test split is clean. The following two blocks handle:

1.  **Cleaning Subdatasets:** Removing Gold IDs from the `subdataset_*.json` files.
2.  **Cleaning Gold Citations:** Stripping the actual `\cite{...}` strings from the gold dataset text to prevent the model from trivially finding citations.

In [ ]:
# Erasing articles used in gold from each subdataset:
for strat_name in strategies.keys():
    print(f"\n=== Cleaning subdataset for strategy: {strat_name} ===")
    
    # 1. Load gold dataset
    gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_{strat_name}.json')
    if not os.path.exists(gold_path):
        print(f"Gold dataset {gold_path} not found, skipping.")
        continue
        
    with open(gold_path, 'r', encoding='utf-8') as f:
        gold_data = json.load(f)
    gold_ids = set([art['id'] for art in gold_data])
    
    # 2. Load subdataset (Structure: {"articles": [...]})
    subdataset_path = os.path.join(OUTPUT_DIR, f'subdataset_{strat_name}.json')
    if not os.path.exists(subdataset_path):
        print(f"Subdataset {subdataset_path} not found, skipping.")
        continue

    with open(subdataset_path, 'r', encoding='utf-8') as f:
        subdataset_data = json.load(f)
        
    original_articles = subdataset_data.get('articles', [])
    
    # 3. Filter out gold articles
    clean_article_list = [art for art in original_articles if art['id'] not in gold_ids]
    
    # 4. Structure the output to match input (Dict wrapper)
    clean_output_data = {
        "articles": clean_article_list
    }
    
    # 5. Save cleaned subdataset
    clean_path = os.path.join(OUTPUT_DIR, f'clean_subdataset_{strat_name}.json')
    with open(clean_path, 'w', encoding='utf-8') as f:
        json.dump(clean_output_data, f, indent=2)
    
    print(f"Cleaned subdataset saved to {clean_path}")
    print(f"Size reduced from {len(original_articles)} to {len(clean_article_list)} articles.")

In [ ]:
# In each gold dataset, we remove any string like \cite{...} to avoid leakage.
for strat_name in strategies.keys():
    print(f"\n=== Cleaning citations in gold dataset for strategy: {strat_name} ===")
    gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_{strat_name}.json')
    if not os.path.exists(gold_path):
        print(f"Gold dataset {gold_path} not found, skipping.")
        continue
    with open(gold_path, 'r') as f:
        gold_data = json.load(f)
    
    # Clean citations in chunks
    for art in gold_data:
        for ref_id, chunks in art['citations'].items():
            cleaned_chunks = []
            for chunk in chunks:
                cleaned_chunk = CITE_PATTERN.sub('', chunk)
                cleaned_chunks.append(cleaned_chunk)
            art['citations'][ref_id] = cleaned_chunks
            
    # Save cleaned gold dataset
    cleaned_gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_cleaned_{strat_name}.json')
    with open(cleaned_gold_path, 'w') as f:
        json.dump(gold_data, f, indent=2)
        
    print(f"Cleaned gold dataset saved to {cleaned_gold_path}.")

In [ ]:
# Cleanup Temp Directory
shutil.rmtree(TEMP_DIR, ignore_errors=True)
print("\nAll tasks completed. Temporary files removed.")

# II - Methods and evalulation
This part is organized into three main sections:
1. **Graph-based Community Detection**: Using Louvain modularity to cluster articles and retrieval via embeddings. This part therefore also includes the embedding method.
2. **Locality Sensitive Hashing (LSH)**: Using MinHash and LSH for fast approximate nearest neighbor search.
3. **Evaluation**: Metrics to assess the performance of these methods.

---

## Part 1: Graph Community Detection & Embedding Retrieval

This method constructs a graph where nodes are articles and edges are citations. Communities are detected to group related papers. Retrieval is a two-step process: finding an entry point via embeddings, then expanding search within the community.

### 1.1 Libraries & Imports
Core libraries for graph processing (NetworkX), embeddings (SentenceTransformers), and vector search (Faiss).

In [1]:
import networkx as nx
from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community.quality import modularity
import json
import matplotlib.pyplot as plt
from operator import itemgetter
from typing import Set, Dict, Any, Tuple
import os
import numpy as np

from operator import itemgetter # Utilisé pour trier

In [6]:
from sentence_transformers import SentenceTransformer

# Define output paths variables to ensure consistency
output_emb_file = f'data/embeddings/embeddings_{d_type}.npy'
output_ids_file = f'data/embeddings/doc_ids_{d_type}.json'

# Check if the output files already exist
if os.path.exists(output_emb_file) and os.path.exists(output_ids_file):
    # Files exist: Skip the expensive computation
    print(f"Embeddings and IDs found at '{output_emb_file}'. Skipping computation.")
else:
    # Files do not exist: Proceed with loading data and computing embeddings
    print("Output files not found. Starting data loading and embedding computation...")

    with open(f"data/processed/clean_subdataset_{d_type}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)

    texts = [article["clean_text"] for article in data["articles"]]
    doc_ids = [article["id"] for article in data["articles"]]

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    # Call the function
    compute_and_save_embeddings(
        texts,
        doc_ids,
        model_name='all-MiniLM-L6-v2',
        batch_size=8,
        out_emb_path=output_emb_file,
        out_ids_path=output_ids_file,
        overwrite=True
    )
    print("Computation finished and files saved.")

# Loading and verification (Load from the specific paths defined above)
print("-" * 30)
print("Verifying loaded data...")

loaded_emb = np.load(output_emb_file, mmap_mode='r')
with open(output_ids_file, 'r', encoding='utf-8') as f:
    loaded_ids = json.load(f)

print("Embeddings shape:", loaded_emb.shape)
print("Number of doc ids:", len(loaded_ids))
print("Example embedding (first doc) first 10 dims:", loaded_emb[0][:10])

/Users/surprisedcat/DTU/DS/DTU_DS_PROJECT_69/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embeddings and IDs found at 'data/embeddings/embeddings_waterfilling.npy'. Skipping computation.
------------------------------
Verifying loaded data...
Embeddings shape: (49980, 384)
Number of doc ids: 49980
Example embedding (first doc) first 10 dims: [-0.09529466 -0.05684136  0.06107605  0.07362463  0.04787469 -0.00985583
 -0.06664003  0.01534571 -0.05632101  0.07219958]


### 1.2 Function Definitions
Here we define all core logic for the graph pipeline:
- `create_graph_from_filtered_json`: Loads data and builds the NetworkX graph.
- `community_detection`: Applies Louvain algorithm.
- `compute_and_save_embeddings`: Generates vector representations of text.
- `graph_retrieval_pipeline`: The main hybrid recommendation function.

In [2]:
d_type = "waterfilling"
FILTERED_JSON_PATH = f"data/processed/clean_subdataset_{d_type}.json"


def create_graph_from_filtered_json(
    json_path: str = FILTERED_JSON_PATH,
) -> nx.DiGraph:
    """
    Builds a directed graph (DiGraph) from the filtered JSON file.
    The graph is created in memory and is NOT saved or loaded from disk.
    """

    print("--- Creating graph from filtered JSON (in memory)... ---")
    
    G = nx.DiGraph()
    
    # 1. Load JSON data (Consolidated try/except)
    try:
        with open(json_path, 'r', encoding='utf-8') as json_file:
            data = json.load(json_file)
            
        articles = data.get('articles', [])
        
        # 2. Graph creation
        for article in articles:
            article_id = article.get('id')
            if article_id is not None:
                G.add_node(article_id) 
                for link in article.get('refs', []):
                    G.add_edge(article_id, link)
                    
             
    except FileNotFoundError:
        print(f"Error: Filtered JSON file not found at: {json_path}. Run 'filter_json_and_save' first.")
        return nx.DiGraph()
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in file: {json_path}")
        return nx.DiGraph()
    except Exception as e:
        print(f"Error during graph creation: {e}")
        return nx.DiGraph()

    print(f"Graph created: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")
    
    return G

In [3]:
def community_detection():
    results = [] 
    G = create_graph_from_filtered_json()


    # Louvain community detection
    communities = louvain_communities(G, seed=42)

    for comm in communities:
        # Identify the node with the highest degree as the representative
        # representative_article = sorted(comm, key=lambda x: G.in_degree(x), reverse=True)[0]
        results.append({
            # 'representative_node' : representative_article,
            # # Convert set to list for JSON serialization
            'community' : sorted(comm, key=lambda x: G.in_degree(x), reverse=True)
        })

    with open(f'data/processed/communities_{d_type}.json', 'w') as outfile:
        # Use indent for better JSON readability
        json.dump(results, outfile, indent=4) 


In [5]:
def compute_and_save_embeddings(texts, doc_ids, model_name='all-mpnet-base-v2',
                                batch_size=64, out_emb_path='embeddings.npy',
                                out_ids_path='doc_ids.json', overwrite=True):
    if os.path.exists(out_emb_path) and not overwrite:
        raise FileExistsError(f"{out_emb_path} already exists. Set overwrite=True to replace.")

    model = SentenceTransformer(model_name)
    n = len(texts)
    emb_dim = model.get_sentence_embedding_dimension()

    emb_memmap = np.lib.format.open_memmap(out_emb_path, mode='w+', dtype='float32', shape=(n, emb_dim))

    for i in tqdm(range(0, n, batch_size), desc="Embedding batches"):
        batch_texts = texts[i:i+batch_size]
        batch_emb = model.encode(batch_texts, show_progress_bar=False, convert_to_numpy=True)
        # normalize rows to unit vectors (for cosine via inner product)
        norms = np.linalg.norm(batch_emb, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        batch_emb = batch_emb / norms
        emb_memmap[i:i+len(batch_emb)] = batch_emb.astype('float32')

    # ensure data flushed to disk
    del emb_memmap

    # save doc ids as JSON
    with open(out_ids_path, 'w', encoding='utf-8') as f:
        json.dump(list(doc_ids), f, ensure_ascii=False)

    print(f"Saved embeddings -> {out_emb_path}")
    print(f"Saved doc ids -> {out_ids_path}")

In [6]:
def build_faiss_index(embeddings_path=f'data/embeddings/embeddings_{d_type}.npy', index_path=f'data/embeddings/faiss_{d_type}.index',
                      index_type='hnsw', ef_construction=200, M=32):
    if os.path.exists(index_path):
        return faiss.read_index(index_path)
    
    emb = np.load(embeddings_path, mmap_mode='r')  # shape (N, d)
    d = emb.shape[1]
    if index_type == 'flat':
        index = faiss.IndexFlatIP(d)  # inner product -> cosine if vectors normalized
        index.add(emb)
    elif index_type == 'hnsw':
        index = faiss.IndexHNSWFlat(d, M)  # M controls connectivity
        index.hnsw.efConstruction = ef_construction
        index.add(emb)
    else:
        raise ValueError('index_type not supported')
    faiss.write_index(index, index_path)
    return index

In [7]:
def retrieve_similar_articles(query, model, embeddings, articles, index, top_n=5, use_ann=False):
    query_emb = model.encode([query], convert_to_numpy=True)
    # normalize
    query_emb = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
    
    if use_ann:
        distances, indices = index.search(query_emb.astype('float32'), top_n)
    else:
        embeddings = np.load(embeddings, mmap_mode='r')
        # Exact search using sklearn
        nbrs = NearestNeighbors(n_neighbors=top_n, metric="cosine").fit(embeddings)
        distances, indices = nbrs.kneighbors(query_emb)
    
    results = pd.DataFrame({
        "article_id": [articles[i] for i in indices[0]],
        "similarity": [1 - d for d in distances[0]]
    })
    return results

In [ ]:
def get_representative(member_id, file_path, top_n=1):
    """
    Get the top_n most cited nodes of the community of member_id.
    If the community is not large enough to have top_n articles, the functions return all the sorted community.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for group in data:
            community = group.get("community", [])
            if member_id in community:
                limit = min(top_n, len(community))
                return community[:limit]
                
        return []

    except (FileNotFoundError, json.JSONDecodeError):
        print("[Get representative]: json file not found or invalid")
        return []

In [ ]:
def graph_retrieval_pipeline(query_text, n, **kwargs):
    """
    Bridge function to allow Graph recommendation on new text.
    Ensure exactly n items are returned if possible.
    
    Requires kwargs:
    - model: SentenceTransformer model
    - embeddings: Path to embeddings of the dataset
    - articles: List of article IDs corresponding to embeddings
    - index: Faiss index of embeddings
    - communities: Path to communities.json
    """
    
    # 1. Retrieve more candidates (e.g., n*3) than necessary via embeddings
    # to ensure enough entry points if clusters are small.
    bridge_result = retrieve_similar_articles(
        query=query_text,
        model=kwargs['model'],
        embeddings=kwargs['embeddings'],
        articles=kwargs['articles'],
        index=kwargs['index'],
        top_n=n * 3, 
        use_ann=True
    )
    
    if bridge_result.empty:
        return []
    
    recommendations = []
    seen_ids = set() # To avoid duplicates
    i = 0
    
    # 2. Loop: while we don't have n recs AND there are remaining bridge candidates
    while len(recommendations) < n and i < len(bridge_result):
        
        # How many articles are missing to reach n?
        needed = n - len(recommendations)
        
        entry_node_id = bridge_result.iloc[i]['article_id']
        
        # Retrieve up to 'needed' candidates from this cluster
        cluster_candidates = get_representative(
            member_id=entry_node_id,
            file_path=kwargs['communities'],
            top_n=needed
        )
        
        # Add candidates while checking for duplicates
        for candidate in cluster_candidates:
            if candidate not in seen_ids:
                recommendations.append(candidate)
                seen_ids.add(candidate)
                
                # If we reach n, stop immediately
                if len(recommendations) == n:
                    break
        
        i += 1
        
    
    # Fallback: If after traversing the graph we have fewer than n articles
    # (very rare case where the graph is empty or disconnected), fill the rest with
    # raw embedding (bridge) results.
    if len(recommendations) < n:
        for idx, row in bridge_result.iterrows():
            cand = row['article_id']
            if cand not in seen_ids:
                recommendations.append(cand)
                seen_ids.add(cand)
            if len(recommendations) == n:
                break
                
    return recommendations

### 1.3 Execution & Tests
The following cells run the pipeline: loading data, building the graph, and testing the retrieval.

In [7]:
community_detection()

--- Creating graph from filtered JSON (in memory)... ---
Graph created: 214793 nodes, 1276385 edges.


In [8]:
file_path = f'data/processed/communities_{d_type}.json'  

with open(file_path, 'r') as infile:
    commu = json.load(infile)
print(len(commu))

article_name = {}
number_citation = {}
with open(FILTERED_JSON_PATH, 'r' ) as infile:
    nodes = json.load(infile)
for article in nodes['articles']:
    article_name[article['id']] = article['title']

36989


In [4]:
# Core Python libraries
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
import re
import os

# NLP and Embeddings
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import faiss


# Visualization and evaluation
import matplotlib.pyplot as plt
import seaborn as sns



/Users/surprisedcat/DTU/DS/DTU_DS_PROJECT_69/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
build_faiss_index()

<faiss.swigfaiss_avx2.IndexHNSWFlat; proxy of <Swig Object of type 'faiss::IndexHNSWFlat *' at 0x0000024AFC06BE70> >

---

## Part 2: Locality Sensitive Hashing (LSH)

This method uses probabilistic hashing to find similar documents quickly. It converts text to shingles, computes MinHash signatures, and uses Banding for candidate generation.

### 2.1 Libraries & Imports
Libraries for hashing (mmh3) and array manipulation.

In [11]:
import mmh3
import json 
from typing import Dict, Any
import numpy as np
from tqdm import tqdm
from mmh3 import hash

### 2.2 Function Definitions
Core LSH logic:
- `shingle`: Converts text to k-shingles.
- `minhash` & `signatures`: Creates compact signature matrices.
- `lsh_band_hash`: Implements the banding technique.
- `lsh`: Main retrieval function using Jaccard similarity.

In [12]:
def find_index(L, x):
    """
    Calculates the index (i) of the first occurrence of element x in list L.

    Args:
        L (list): The list to search within.
        x (any): The element whose index is being sought.

    Returns:
        int: The index of element x in L.

    Raises:
        ValueError: If element x is not found in list L.
    """
    try:
        # The index() method returns the index of the first occurrence
        # of the specified element.
        index = L.index(x)
        return index
    except ValueError:
        # index() raises a ValueError if the element is not found.
        # It's good practice to handle this error.
        raise ValueError(f"The element '{x}' is not in the list.")

In [13]:
punctuation = ['.',',',';',':','"']


def string_modulo_q(q, caracter_list, beginning_index):
    """takes a caracter list and a beginning index and returns a string composed of 
     the caracters in order beginning at the beginning index """
    assert q == len(caracter_list)

    string=""

    for i in range(q):
        string += caracter_list[(beginning_index+i)%q]

    return string


def shingle(q, text, punctuation_list = punctuation):
    "Returns the set of q-shingles from the original text"
    n = len(text)
    assert(n>q), "Text too short or shingle too long"
    assert q>=2, "Shingle size must be > 1 caracter "

    ind = 0 # index of the first caracter
    q_list = ["" for _ in range(q)]
    S = []
    ind_modified = 0

    for c in text:
        if (not (c in punctuation_list)) & (c != ' ') : # We ignore punctuation
            
            # Update q_string with another caracter
            q_list[ind] = c
            # Add one to the beginning index
            new_ind = (ind+1)%q
            ind = new_ind
            ind_modified += 1
            # New shingle added to the list
            S.append(string_modulo_q(q = q, caracter_list = q_list, beginning_index = ind))
    return S[q-1:]

In [14]:
# hashes a list of strings
def listhash(l,seed):
	val = 0
	for e in l:
		val = val ^ hash(e, seed)
	return val 

# Minhash function
def minhash(shingle_list, k):
    "returns a list of k different minhashes of the shingle list"
    return [min(listhash(s, seed) for s in shingle_list) for seed in range(k)]


In [27]:
def signatures(doc_list, shingle_size, signature_size):
    """
    inputs :
        - doc_lists : list of document of the form {'id': ... , 'abstract' : ... }
        - signature_size : size of the signatures
    outputs :
        - sig : signature matrix, every column represent the signature of a document
        - idx_to_sig : dictionnary that matches idexes (columns of sig) withs ids of the documents
    """
    idx_to_id = {} # dictionary of the signatures of each document
    n = len(doc_list)
    sig = np.zeros((signature_size,n))
    # signature of doc no "id" using minhashing on the shingle_list. Size of the shingles is shingle_size
    for i in range(n):
        id = doc_list[i]["id"]
        abstract = doc_list[i]["clean_text"]
        idx_to_id[i] = id
        sig[:,i] = np.array(minhash(shingle(q=shingle_size, text=abstract), k=signature_size))
    return sig, idx_to_id

In [16]:
def lsh_band_hash(band, m, lsh_seed) -> int:
    """
    Computes a hash value for a single band (a list of r integers).
    The goal is to map identical bands to the same hash bucket.

    band: A list of r integers representing the signature's portion 
            for a specific band.
    Returns: An integer hash value for the band.
    """
    
    band_string = ",".join(map(str, band)) # Convert the band to a string representation.
    
    hash_value = mmh3.hash(band_string, lsh_seed) # Compute the hash using MurmurHash3.
    
    return abs(hash_value) % m # Ensure non-negative and fit within m buckets

In [17]:
def Jaccard_similarity_signatures(input_signature, doc_signature) -> float :
    "Returns an approximation of the jaccard similarity between 2 documents doc_name1 and doc_name2 using signatures"
    sig = doc_signature 
    S = 0
    k = len(sig) # size of signature list
    assert k == len(input_signature), "Signatures are not matching size"
    for i in range(k): # Loop over the signature pairs from the two documents
        if sig[i]==input_signature[i]:
            S+=1
    return S/k

def Jaccard_similarity_shingles(shingles_list_A, shingles_list_B):
    """
    Calculates the Jaccard similarity coefficient between two lists of shingles.

    Args:
        shingles_list_A (list): The list of shingles for the first document.
        shingles_list_B (list): The list of shingles for the second document.

    Returns:
        float: The Jaccard similarity score (0.0 to 1.0).
    """

    # 1. Convert lists to sets for efficient set operations and to ensure uniqueness
    set_A = set(shingles_list_A)
    set_B = set(shingles_list_B)

    # 2. Calculate the size of the intersection (common shingles)
    intersection_size = len(set_A.intersection(set_B))
    
    # Alternatively: intersection_size = len(set_A & set_B)

    # 3. Calculate the size of the union (all unique shingles combined)
    union_size = len(set_A.union(set_B))
    
    # Alternatively: union_size = len(set_A | set_B)

    # 4. Calculate the Jaccard score
    if union_size == 0:
        # Avoid division by zero if both lists are empty
        return 0.0

    jaccard_score = intersection_size / union_size
    return jaccard_score


def Jaccard_similarity(input_text ,article_list, candidate_idx, q):
    return Jaccard_similarity_shingles(
        shingles_list_A= shingle(q = q, text = input_text),
        shingles_list_B= shingle(q = q, text = article_list[candidate_idx]["clean_text"]))

In [18]:
def Jaccard_similarity_signatures(input_signature, doc_signature) -> float :
    "Returns an approximation of the jaccard similarity between 2 documents doc_name1 and doc_name2 using signatures"
    sig = doc_signature 
    S = 0
    k = len(sig) # size of signature list
    assert k == len(input_signature), "Signatures are not matching size"
    for i in range(k): # Loop over the signature pairs from the two documents
        if sig[i]==input_signature[i]:
            S+=1
    return S/k

In [19]:
def Jaccard_similarity_shingles(shingles_list_A, shingles_list_B):
    """
    Calculates the Jaccard similarity coefficient between two lists of shingles.

    Args:
        shingles_list_A (list): The list of shingles for the first document.
        shingles_list_B (list): The list of shingles for the second document.

    Returns:
        float: The Jaccard similarity score (0.0 to 1.0).
    """

    # 1. Convert lists to sets for efficient set operations and to ensure uniqueness
    set_A = set(shingles_list_A)
    set_B = set(shingles_list_B)

    # 2. Calculate the size of the intersection (common shingles)
    intersection_size = len(set_A.intersection(set_B))
    
    # Alternatively: intersection_size = len(set_A & set_B)

    # 3. Calculate the size of the union (all unique shingles combined)
    union_size = len(set_A.union(set_B))
    
    # Alternatively: union_size = len(set_A | set_B)

    # 4. Calculate the Jaccard score
    if union_size == 0:
        # Avoid division by zero if both lists are empty
        return 0.0

    jaccard_score = intersection_size / union_size
    return jaccard_score


def Jaccard_similarity(input_text ,article_list, candidate_idx, q):
    return Jaccard_similarity_shingles(
        shingles_list_A= shingle(q = q, text = input_text),
        shingles_list_B= shingle(q = q, text = article_list[candidate_idx]["clean_text"]))

In [20]:
# Preprocessing the dataset to keep only the relevant information

def preprocess_lsh(dataset_path):
    """
    Input : 
        dataset_path : path of the json dataset of articles 
    Output :
        article_list : list of dictionnaries of the form {'id': ..., 'abstract': ...}  
    """
    try : 
        with open(dataset_path, 'r', encoding='utf-8') as f:
            data: Dict[str, Any] = json.load(f)
        print(f"Data succesfully loaded")
        
        article_list = [{'id': article['id'], 'clean_text' : article['clean_text']} for article in data] 
        return article_list

    #data = {'articles' : [d1 = {'id' : ..., 'authors' : ...,'abstract': ..., 'clean_text' : ... , 'categories' : ... , 'refs' :  ... } , d2, ...]}

    except Exception as e:
        print(f"Error in loading of the dataset : {e}")


In [21]:

# Implementing lsh function         

def lsh(input, article_list ,signature_matrix, idx_to_id, m, shingle_size, nb_band, band_size):
    """
    Inputs :
        input : input text from which we want to obtain sources
        article_list : list of arcticles i.e. dictionnaries of the format {'id': ..., 'abstract': ...}
        shingle_size : size of the shingle decomposition on which the minhashing is computed
        nb_band : number of horizontal bands in the signature matrix 
        band_size : number of rows per band in the signature matrix
        signature_size : size of the signatures of the documents obtained from minhashing of
                        the shingle size with signature_size different seeds. 
                        signature_size = nb_band*band_size

    Process :
        - Shingle all documents from the dataset and compute a signature for every document
          using minhashing (signatures function)
        - Find the documents that are most likely to be similar to input using LSH method
        - Compute the actual similarity between input and these document to eliminate false positives
    
    Outputs :
        - Most_similar : list of the most similar documents
        - Scores : list of Jaccard_similarities between input and documents 
    
    """

    # 1. Compute the a signature for every document

    k = band_size*nb_band # Signature size

    # Compute signature of input
    input_signature = signatures([{'id':'input', 'clean_text':input}],
                                 shingle_size = shingle_size,
                                 signature_size = k)[0][:,0]
    
    # 2. Find the documents that are most likely to be similar to input using LSH method

    similar_candidates = {}
    n = len(signature_matrix[0])  # number articles in the dataset

    for band_nb in range(nb_band):
        input_hash = lsh_band_hash(
            band = input_signature[band_nb*band_size:(band_nb+1)*band_size],
            m = m,
            lsh_seed = band_nb
        )
        for i in range(n):
            doc_hash = lsh_band_hash(
                band = signature_matrix[:,i][band_nb*band_size:(band_nb+1)*band_size],
                m = m,
                lsh_seed = band_nb
                )
            if doc_hash == input_hash :
                if i in similar_candidates :
                    similar_candidates[i] += 1
                else :
                    similar_candidates[i] = 1


    # 3. Compute the actual similarity between input and these documents


    Ordered_similar_candidates = similar_candidates.keys()
    Ordered_similarities = []
    for idx in similar_candidates :
        ## In case we want to compare the candidates on the jaccard similarities of the shingle lists
        #j = Jaccard_similarity(input_text= input, 
        #                       article_list= article_list,
        #                       candidate_idx=idx,
        #                       q=shingle_size)
        j = Jaccard_similarity_signatures(input_signature=input_signature, doc_signature=signature_matrix[:,idx])
        Ordered_similarities.append(j)
    Most_similar = zip(Ordered_similar_candidates,Ordered_similarities)
    Most_similar = sorted(Most_similar, key = lambda pair:pair[1], reverse = True)
    Scores = [ p[1] for p in Most_similar]
    Most_similar = [idx_to_id[p[0]]['id'] for p in Most_similar]
    
    return Most_similar, Scores


In [22]:
def lsh_n(n,input, article_list ,signature_matrix, idx_to_id, m, shingle_size, nb_band, band_size) : 
    """ Returns the Most_similar list of lsh with the n most relevant results only. If the number of result from lsh 
    is < n, random articles from the dataset are added"""

    Most_similar = lsh(input, article_list ,signature_matrix, idx_to_id, m, shingle_size, nb_band, band_size)[0]
    length = len(Most_similar)
    if length < n :
        c = 0
        while length < n :
            extra_id = article_list[c]['id']
            if extra_id not in Most_similar :
                Most_similar.append(extra_id)
                length += 1
            c += 1    
    return Most_similar[:n]

In [28]:
def calculate_top_n_accuracy(method_name, top_n_list, gold_set_path, **kwargs):
    """
    Calculates Top-N Accuracy for multiple N values simultaneously.
    
    Args:
        method_name (str): 'embedding', 'lsh', or 'graph'.
        top_n_list (list): List of integers for which to calculate accuracy (e.g. [1, 5, 10]).
        gold_set_path (str): Path to the .json gold set file.
        **kwargs: Variable arguments required for the specific methods.

    Returns:
        dict: A dictionary where keys are N and values are the accuracy scores.
    """
    
    # 1. Load Gold Set
    try:
        with open(gold_set_path, 'r', encoding='utf-8') as f:
            gold_set = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading gold set: {e}")
        return {}

    # 2. Determine the maximum K needed
    max_k = max(top_n_list)
    
    # Initialize counters for each k in the list
    hits_at_k = {k: 0 for k in top_n_list}
    
    # Pre-calculate total samples
    total_samples = 0
    for paper in gold_set:
        citations = paper.get('citations', {})
        for context_list in citations.values():
            total_samples += len(context_list)

    if total_samples == 0:
        return {k: 0.0 for k in top_n_list}

    print(f"--- Starting evaluation for {method_name} (Max K={max_k}) ---")
    pbar = tqdm(total=total_samples, desc="Processing Chunks")

    # 3. Main Loop
    for paper_entry in gold_set:
        citations = paper_entry.get('citations', {})
        
        for true_id, text_chunks in citations.items():
            for query_text in text_chunks:
                pbar.update(1)
                
                if not query_text or not query_text.strip():
                    continue

                # RETRIEVAL STEP
                recommended_ids = []
                
                try:
                    if method_name == "embedding":
                        df_results = retrieve_similar_articles(
                            query=query_text,
                            model=kwargs['model'],
                            embeddings=kwargs['embeddings'],
                            articles=kwargs['articles'],
                            index=kwargs['index'],
                            top_n=max_k,
                            use_ann=kwargs.get('use_ann', False)
                        )
                        recommended_ids = df_results['article_id'].tolist()

                    elif method_name == "LSH":
                        recommended_ids = lsh_n(
                            n=max_k,
                            input=query_text, 
                            article_list=kwargs['article_list'],
                            signature_matrix=kwargs['signature_matrix'],
                            idx_to_id=kwargs['idx_to_id'],
                            m=kwargs['m'],
                            shingle_size=kwargs['shingle_size'],
                            nb_band=kwargs['nb_band'],
                            band_size=kwargs['band_size'],
                        )

                    elif method_name == "graph":
                        recommended_ids = graph_retrieval_pipeline(
                            query_text=query_text,
                            n=max_k,
                            model=kwargs['model'],
                            embeddings=kwargs['embeddings'],
                            articles=kwargs['articles'],
                            index=kwargs['index'],
                            communities=kwargs['communities']
                        )
                    else:
                        raise ValueError(f"Unknown method: {method_name}")
                except Exception as e:
                    print(f"Error processing query: {e}") 
                    recommended_ids = []

                # METRIC CALCULATION STEP
                true_id_str = str(true_id)
                rec_ids_str = [str(x) for x in recommended_ids]
                
                # Check rank
                try:
                    rank = rec_ids_str.index(true_id_str)
                    position = rank + 1
                    
                    # Update counters:
                    for k in top_n_list:
                        if position <= k:
                            hits_at_k[k] += 1
                            
                except ValueError:
                    pass
    
    pbar.close()

    # 4. Final Calculation
    accuracies = {k: hits / total_samples for k, hits in hits_at_k.items()}
    
    return accuracies

### 2.3 Execution & Tests
Running the LSH pipeline: Preprocessing the dataset, computing signatures, and finding nearest neighbors.

In [ ]:
datatypes = ["most_cited", "quartiles", "stratified", "waterfilling"]
method = 2

In [23]:
json_path = f"data/clean_subdataset_{datatypes[method]}.json" 
data = preprocess_lsh(dataset_path = json_path)
print("preprocessing done")

Data succesfully loaded
preprocessing done


In [24]:
print(data[0])

{'id': '1404.3723', 'clean_text': 'we highlight the progress current status and open challenges of qcddriven\nphysics in theory and in experiment we discuss how the strong interaction is\nintimately connected to a broad sweep of physical problems in settings ranging\nfrom astrophysics and cosmology to stronglycoupled complex systems in\nparticle and condensedmatter physics as well as to searches for physics\nbeyond the standard model we also discuss how success in describing the strong\ninteraction impacts other fields and in turn how such subjects can impact\nstudies of the strong interaction in the course of the work we offer a\nperspective on the many research streams which flow into and out of qcd as\nwell as a vision for future developments'}


In [25]:
output_json_path = f'data/subdataset_lsh_{datatypes[method]}.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)

In [ ]:
data_path = f'data/subdataset_lsh_{datatypes[method]}.json'
with open(data_path, 'r', encoding='utf-8') as f:
            data: Dict[str, Any] = json.load(f)

: 

In [ ]:
q = 7 
b = 10
r = 10

signature_matrix_Nmost, idx_to_id = signatures(
    doc_list=data,
    shingle_size = q,
    signature_size = b*r
    )

Computing signatures:  17%|█▋        | 8529/49980 [1:49:58<1:20:07,  8.62it/s]      

In [ ]:
np.save(file = f"data/signature_lsh_{datatypes[method]}_q{q}_b{b}_r{r}", arr = signature_matrix_Nmost)
output_json_path = f"data/idx_to_id_lsh_{datatypes[method]}_q{q}_b{b}_r{r}.json"
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)

In [26]:
data_path = f'data/subdataset_lsh_{datatypes[method]}.json'
with open(data_path, 'r', encoding='utf-8') as f:
            data: Dict[str, Any] = json.load(f)

q = 7 
b = 10
r = 10

try :
    signature_matrix = np.load(f"data/signature_lsh_{datatypes[method]}_q{q}_b{b}_r{r}.npy")
    with open(f"data/idx_to_id_lsh_{datatypes[method]}_q{q}_b{b}_r{r}.json", 'r', encoding='utf-8') as f:
            idx_to_id: Dict[str, Any] = json.load(f)
except FileNotFoundError as e :
    print(f"No signature matrix has been saved with the set of parameters : q = {q}, b = {b}, r = {r} ")

In [27]:
Most_similar_100, Scores = lsh(
        input = data[0]['clean_text'],
        article_list=data,
        signature_matrix = signature_matrix,
        idx_to_id = idx_to_id,
        shingle_size=q,
        m = signature_matrix.shape[1]//10, 
        nb_band = b,
        band_size = r,
        )

print("Most similar documents : " , Most_similar_100, '\n')
print("Scores : " , Scores)

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures: 100%|██████████| 1/1 [00:00<00:00,  8.60it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands: 100%|██████████| 10/10 [00:04<00:00,  2.02it/s]

LSH successfully performed to find similar candidates
Calculation of the actual similarities ... 

Most similar documents :  ['1404.3723', 'cond-mat/0402568', '1501.03060', '1407.4706', '1805.09127', '1811.06447', '1103.5690', '1011.6122', '1710.09302', '1310.5637', '1805.01853', 'math/0106271', '1401.6020', 'math/9811160', '0709.0668', '1704.01581', '1409.0356', '1802.05149', '1804.03579', '1307.8026', '1210.2058', '0908.1375', '1010.5456', '0710.0787', '1612.01216', '1809.11156', '1302.0134', 'math/0011227', '1505.03044', '1311.4290', '1004.2355', '1010.2065', '1609.03820', '1605.07473', '1603.00097', '1802.00844', '0802.3897', '1603.00710', '1604.06776', '1304.3772', '1211.0665', '1706.09869', '1812.03453', 'math/0202183', '1901.10231', 'cond-mat/0007400', '1812.08099', '1506.07540', '1802.04051', '1212.0133', '1508.02826', '0810.1622', '1604.08179', '0908.4353', '2104.05636', '1603.06914', '1807.09459', '1703.10505', '1707.04896', '1511.00367', '1612.03350', '1806.01110', '1310.574

In [28]:
n=10
Most_similar_100_n = lsh_n(
    n,
    input = data[0]['clean_text'],
        article_list=data,
        signature_matrix = signature_matrix,
        idx_to_id = idx_to_id,
        m = signature_matrix.shape[1]//10, 
        shingle_size = q,
        nb_band = b,
        band_size = r,
)

print(f"{n} most similar documents : ", Most_similar_100_n )

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures: 100%|██████████| 1/1 [00:00<00:00, 11.21it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands: 100%|██████████| 10/10 [00:04<00:00,  2.16it/s]

LSH successfully performed to find similar candidates
Calculation of the actual similarities ... 

10 most similar documents :  ['1404.3723', 'cond-mat/0402568', '1501.03060', '1407.4706', '1805.09127', '1811.06447', '1103.5690', '1011.6122', '1710.09302', '1310.5637']


---

## Part 3: Evaluation

Comparing the performance of Graph vs. LSH approaches using Recall@N metrics.

### 3.2 Evaluation Functions
Helper functions to calculate metrics and compare against gold standard datasets.

In [10]:
def common_commu(result_id, target_id):
    # are the result_id and the tardget_id in the same community / cluster ?
    COM_FILE = f'data/processed/communities_{d_type}.json'
    with open(COM_FILE, 'r') as file:
        commu = json.load(file)
    
    for commu_l in commu:
        if result_id in commu_l['community'] and target_id in commu_l['community']:
            return True
    
    return False

### 3.3 Results Visualization
Running the evaluation loops and plotting the results.

In [ ]:
# --- Configuration ---
top_n_list = list(range(1, 21))
dataset_types = ["quartiles", "stratified"]

# Static paths (global)
MODEL_NAME = 'all-MiniLM-L6-v2'

# --- Load global resources (done once) ---
print("Loading SentenceTransformer model and global Embeddings...")
model = SentenceTransformer(MODEL_NAME)

# Initialize global results dictionary
# Structure: global_results[type][method][n] = accuracy
global_results = {}

for d_type in dataset_types:
    print(f"\n{'='*60}")
    print(f"PROCESSING DATASET: {d_type.upper()}")
    print(f"{'='*60}")
    
    global_results[d_type] = {}
    
    # --- 1. Construct dynamic paths ---
    gold_set_path = f"data/goldset/gold_dataset_cleaned_{d_type}.json"
    article_list_path = f"data/processed/clean_subdataset_{d_type}.json"
    
    # LSH paths
    sig_matrix_path = f"data/lsh/signature_lsh_{d_type}_q7_b10_r10.npy"
    idx_to_id_path = f"data/lsh/idx_to_id_lsh_{d_type}_q7_b10_r10.json"

    # Embeddings paths
    embeddings = f"data/embeddings/embeddings_{d_type}.npy"
    article_ids = f"data/embeddings/doc_ids_{d_type}.json"
    index_path = f"data/embeddings/faiss_{d_type}.index"

    # Graph paths
    communities = f"data/processed/communities_{d_type}.json"
    
    # --- 2. Load dataset-specific data ---
    try:
        # Load article list (needed to map ID <-> Index)
        with open(article_list_path, 'r') as f:
            article_list_raw = json.load(f)
        article_list_data = article_list_raw['articles']

        # Load LSH matrices
        signature_matrix = np.load(sig_matrix_path)
        
        # Load idx_to_id
        with open(idx_to_id_path, 'r') as f:
            idx_to_id = json.load(f)

        # Parameters for lsh
        m = signature_matrix.shape[0]//10
        nb_band = 10
        band_size = 10

        # IDs for embedding
        article_ids_path = f"data/embeddings/doc_ids_{d_type}.json"
        with open(article_ids_path, 'r') as f:
            article_ids_list = json.load(f)
        
        # Faiss index for embedding
        current_faiss_index = faiss.read_index(index_path)
        print(f"FAISS Index loaded into memory for {d_type}.")

        
    except Exception as e:
        print(f"FATAL ERROR loading data for {d_type}: {e}")
        continue

    # --- 3. Method Evaluation ---
    
    # # A. Embedding Method
    # # --------------------
    print(f"\n--- Embedding Evaluation ({d_type}) ---")
    acc = calculate_top_n_accuracy(
        method_name="embedding",
        top_n_list=top_n_list,
        gold_set_path=gold_set_path,
        model=model,
        embeddings=embeddings,
        articles=article_ids_list,
        index=current_faiss_index,
        use_ann=True
    )
    global_results[d_type]['embedding'] = acc

    # # B. Graph Method (Hybrid)
    # # -------------------------
    print(f"\n--- Graph Evaluation ({d_type}) ---")
    acc = calculate_top_n_accuracy(
        method_name="graph",
        top_n_list=top_n_list,
        gold_set_path=gold_set_path,
        model=model,
        embeddings=embeddings,
        articles=article_ids_list,
        index=current_faiss_index,
        communities=communities
    )
    global_results[d_type]['graph'] = acc

    # C. LSH Method
    # --------------
    print(f"\n--- LSH Evaluation ({d_type}) ---")
    acc = calculate_top_n_accuracy(
        method_name="LSH",
        top_n_list=top_n_list,
        gold_set_path=gold_set_path,
        article_list=article_list_data,
        signature_matrix=signature_matrix,
        idx_to_id=idx_to_id,
        m=m,
        shingle_size=7,
        nb_band=nb_band,
        band_size=band_size
    )
    global_results[d_type]['LSH'] = acc

print("\nEvaluation Finished!")
print("\n--- Accuracy @ 5 Summary ---")
for dtype, methods in global_results.items():
    print(f"Dataset: {dtype}")
    for method, scores in methods.items():
        print(f"  - {method}: {scores.get(5, 0):.4f}")

# Save results
with open("evaluation_results_all_methods.json", "w") as f:
    json.dump(global_results, f, indent=4)

Chargement du modèle SentenceTransformer et des Embeddings globaux...

TRAITEMENT DU DATASET : QUARTILES
Index FAISS chargé en mémoire pour quartiles.

--- Évaluation LSH (quartiles) ---
--- Starting evaluation for LSH (Max K=20) ---












































































Processing Chunks:   0%|          | 22/7098 [04:31<24:17:56, 12.36s/it]

























































































































































































































































































































































































































































































































































































































































































































































































































































































TRAITEMENT DU DATASET : STRATIFIED
Index FAISS chargé en mémoire pour stratified.

--- Évaluation LSH (stratified) ---
--- Starting evaluation for LSH (Max K=20) ---


Processing Chunks: 100%|██████████| 5945/5945 [3:35:47<00:00,  2.18s/it]  


Evaluation Terminée !

--- Résumé Accuracy @ 5 ---
Dataset: quartiles
  - LSH: 0.0011
Dataset: stratified
  - LSH: 0.0013
